In [17]:
import nltk
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load the embedding model (downloads ~90MB first time, then cached)
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded ✅")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded ✅


In [18]:
# These simulate the "retrieved documents" a RAG system would fetch
# In a real RAG pipeline, these come from your vector DB retrieval step

source_docs = [
    """The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars 
    in Paris, France. It was constructed between 1887 and 1889 as the centerpiece 
    of the 1889 World's Fair. The tower was designed and built by Alexandre Gustave Eiffel, 
    a French civil engineer. It stands 330 metres tall and is one of the most recognizable 
    structures in the world. The tower receives approximately 7 million visitors per year.""",

    """Photosynthesis is a biological process by which green plants and some other organisms 
    convert light energy into chemical energy stored in glucose. The process takes place 
    primarily in the chloroplasts of plant cells, using chlorophyll to absorb sunlight. 
    Carbon dioxide from the air and water from the soil are the raw materials. 
    The byproducts of photosynthesis include oxygen, which is released into the atmosphere. 
    This process is fundamental to life on Earth.""",

    """Python is a high-level, general-purpose programming language created by Guido van Rossum. 
    The first version was released in 1991. Python emphasizes code readability and simplicity, 
    making it accessible to beginners and professionals alike. It has become one of the most 
    popular languages for data science, machine learning, and artificial intelligence applications. 
    Python supports multiple programming paradigms including procedural, object-oriented, 
    and functional programming."""
]

print(f"Total source documents: {len(source_docs)}")

Total source documents: 3


In [19]:
def split_into_sentences(docs: list[str]) -> list[dict]:
    """
    Takes a list of documents.
    Returns a flat list of sentences with their source doc index.
    
    Returns:
        [{"sentence": "...", "doc_index": 0}, ...]
    """
    all_sentences = []
    
    for doc_idx, doc in enumerate(docs):
        sentences = nltk.sent_tokenize(doc.strip())
        for sent in sentences:
            sent = sent.strip()
            if len(sent) > 10:  # filter out very short fragments
                all_sentences.append({
                    "sentence": sent,
                    "doc_index": doc_idx
                })
    
    return all_sentences


# Test it
sentences = split_into_sentences(source_docs)
print(f"Total sentences extracted: {len(sentences)}\n")
for i, s in enumerate(sentences):
    print(f"[{i}] (Doc {s['doc_index']}) {s['sentence']}")

Total sentences extracted: 15

[0] (Doc 0) The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars 
    in Paris, France.
[1] (Doc 0) It was constructed between 1887 and 1889 as the centerpiece 
    of the 1889 World's Fair.
[2] (Doc 0) The tower was designed and built by Alexandre Gustave Eiffel, 
    a French civil engineer.
[3] (Doc 0) It stands 330 metres tall and is one of the most recognizable 
    structures in the world.
[4] (Doc 0) The tower receives approximately 7 million visitors per year.
[5] (Doc 1) Photosynthesis is a biological process by which green plants and some other organisms 
    convert light energy into chemical energy stored in glucose.
[6] (Doc 1) The process takes place 
    primarily in the chloroplasts of plant cells, using chlorophyll to absorb sunlight.
[7] (Doc 1) Carbon dioxide from the air and water from the soil are the raw materials.
[8] (Doc 1) The byproducts of photosynthesis include oxygen, which is released into the atmosph

In [20]:
def build_faiss_index(sentences: list[dict]):
    """
    Takes a list of sentence dicts.
    Embeds all sentences and stores them in a FAISS index.
    
    Returns:
        index     → FAISS index for similarity search
        sentences → original sentence list (to look up by index)
    """
    
    # Extract just the text for embedding
    texts = [s["sentence"] for s in sentences]
    
    print(f"Embedding {len(texts)} sentences...")
    embeddings = model.encode(texts, show_progress_bar=True)
    
    # Normalize for cosine similarity
    embeddings = embeddings / np.linalg.norm(
        embeddings, axis=1, keepdims=True
    )
    
    # Build FAISS index
    dimension = embeddings.shape[1]  # 384 for MiniLM
    index = faiss.IndexFlatIP(dimension)  # Inner Product = cosine on normalized vectors
    index.add(embeddings.astype('float32'))
    
    print(f"FAISS index built ✅")
    print(f"Total vectors stored: {index.ntotal}")
    
    return index, sentences


# Build the index
faiss_index, sentence_store = build_faiss_index(sentences)

Embedding 15 sentences...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index built ✅
Total vectors stored: 15


In [21]:
def retrieve_span(claim: str, index, sentence_store: list[dict], top_k: int = 3) -> dict:
    """
    Retrieves top 3 spans and returns the best one.
    Also flags low confidence retrievals.
    """
    
    claim_embedding = model.encode([claim])
    claim_embedding = claim_embedding / np.linalg.norm(
        claim_embedding, axis=1, keepdims=True
    )
    
    scores, indices = index.search(claim_embedding.astype('float32'), top_k)
    
    best_idx = indices[0][0]
    best_score = float(scores[0][0])
    best_sentence = sentence_store[best_idx]
    
    return {
        "sentence": best_sentence["sentence"],
        "doc_index": best_sentence["doc_index"],
        "similarity_score": round(best_score, 4),
        "low_confidence": best_score < 0.70,      # ← new flag
        "top_3_spans": [                           # ← now visible for debugging
            {
                "sentence": sentence_store[indices[0][i]]["sentence"],
                "score": round(float(scores[0][i]), 4)
            }
            for i in range(top_k)
        ]
    }

In [22]:
test_claims = [
    "The Eiffel Tower was built in 1889",
    "The Eiffel Tower stands 330 metres tall",
    "The Eiffel Tower was designed by Gustave Eiffel",
    "The Eiffel Tower is located in Paris, France",
    "Photosynthesis occurs in the chloroplasts",
    "Oxygen is released as a byproduct of photosynthesis",
    "Python was created by Guido van Rossum",
    "Python was first released in 1991",
    "Python is widely used in data science"
]

print("SPAN RETRIEVAL RESULTS")
print("="*60)

for claim in test_claims:
    result = retrieve_span(claim, faiss_index, sentence_store)
    
    # confidence indicator
    confidence_label = "⚠️ LOW CONFIDENCE" if result["low_confidence"] else "✅ CONFIDENT"
    
    print(f"\n📌 CLAIM: {claim}")
    print(f"📄 BEST SPAN: {result['sentence']}")
    print(f"📊 SIMILARITY: {result['similarity_score']}  {confidence_label}")
    print(f"📁 FROM DOC: {result['doc_index']}")
    
    # Show top 3 alternatives so we can see what was missed
    print(f"🔍 TOP 3 CANDIDATES:")
    for i, span in enumerate(result["top_3_spans"]):
        print(f"   {i+1}. [{span['score']}] {span['sentence'][:80]}...")
    
    print("-"*60)

SPAN RETRIEVAL RESULTS

📌 CLAIM: The Eiffel Tower was built in 1889
📄 BEST SPAN: The tower was designed and built by Alexandre Gustave Eiffel, 
    a French civil engineer.
📊 SIMILARITY: 0.7728  ✅ CONFIDENT
📁 FROM DOC: 0
🔍 TOP 3 CANDIDATES:
   1. [0.7728] The tower was designed and built by Alexandre Gustave Eiffel, 
    a French civi...
   2. [0.7023] The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars 
 ...
   3. [0.4124] It was constructed between 1887 and 1889 as the centerpiece 
    of the 1889 Wor...
------------------------------------------------------------

📌 CLAIM: The Eiffel Tower stands 330 metres tall
📄 BEST SPAN: It stands 330 metres tall and is one of the most recognizable 
    structures in the world.
📊 SIMILARITY: 0.6662  ⚠️ LOW CONFIDENCE
📁 FROM DOC: 0
🔍 TOP 3 CANDIDATES:
   1. [0.6662] It stands 330 metres tall and is one of the most recognizable 
    structures in...
   2. [0.6616] The Eiffel Tower is a wrought-iron lattice tower located o

In [23]:
# Let's test an edge case — a claim that is slightly WRONG
# The answer said "Gustave Eiffel" but doc says "Alexandre Gustave Eiffel"
# The span retriever should still FIND the right sentence
# The NLI model (Day 3) will then catch the subtle contradiction

edge_case_claims = [
    "The Eiffel Tower was designed by Gustave Eiffel",      # slightly wrong name
    "Photosynthesis requires only sunlight to work",         # wrong claim
    "Python was created in 2005",                            # wrong year
]

print("EDGE CASE TESTING")
print("="*60)
print("NOTE: Span Retriever finds the CLOSEST sentence.")
print("The NLI model (Day 3) will judge if it actually supports the claim.\n")

for claim in edge_case_claims:
    result = retrieve_span(claim, faiss_index, sentence_store)
    print(f"\n⚠️  CLAIM: {claim}")
    print(f"📄 BEST SPAN: {result['sentence']}")
    print(f"📊 SIMILARITY: {result['similarity_score']}")

EDGE CASE TESTING
NOTE: Span Retriever finds the CLOSEST sentence.
The NLI model (Day 3) will judge if it actually supports the claim.


⚠️  CLAIM: The Eiffel Tower was designed by Gustave Eiffel
📄 BEST SPAN: The tower was designed and built by Alexandre Gustave Eiffel, 
    a French civil engineer.
📊 SIMILARITY: 0.8877



⚠️  CLAIM: Photosynthesis requires only sunlight to work
📄 BEST SPAN: Photosynthesis is a biological process by which green plants and some other organisms 
    convert light energy into chemical energy stored in glucose.
📊 SIMILARITY: 0.6737

⚠️  CLAIM: Python was created in 2005
📄 BEST SPAN: Python is a high-level, general-purpose programming language created by Guido van Rossum.
📊 SIMILARITY: 0.7206


In [24]:
print("""
✅ Day 2 Complete!

Now copy the following functions into verifaith/span_retriever.py:
  → split_into_sentences()
  → build_faiss_index()
  → retrieve_span()

Day 3: We build the NLI Entailment Checker using DeBERTa
       That's where contradictions get CAUGHT 🎯
""")


✅ Day 2 Complete!

Now copy the following functions into verifaith/span_retriever.py:
  → split_into_sentences()
  → build_faiss_index()
  → retrieve_span()

Day 3: We build the NLI Entailment Checker using DeBERTa
       That's where contradictions get CAUGHT 🎯

